# Phương pháp xét duyệt cây MiniMax

Notebook này phục vụ nghiên cứu và thực hành **Chương 4 & Chương 5**:
1. **Nạp môi trường & Cài đặt hàm MiniMax cốt lõi** theo nguyên lý NegaMax/Backward Induction.
2. **Visualizer cây tìm kiếm MiniMax dạng Đồ họa (Plot)**: Vẽ sơ đồ cây hoàn chỉnh cho bài toán đơn giản (**Coin Game**) và thế tàn cuộc của **Tic Tac Toe**.
3. **Tỉa cây (Tree Pruning)**: So sánh hiệu năng giữa duyệt vét cạn toàn bộ, **Cắt tỉa độ sâu (Depth Pruning)** và **Cắt tỉa Alpha-Beta (Alpha-Beta Pruning)**.

## 1. Môi trường MiniMax & Thuật toán Nền tảng

Trong Zero-Sum Game, ta áp dụng công thức NegaMax: `my_payoff = -opponent_payoff`. Điểm số của đối thủ chính là giá trị đối nghịch với điểm của ta.

In [ ]:
import numpy as np
from copy import deepcopy
import random
import time

# Nạp môi trường từ thư mục utils nội bộ
from utils.coin_simple_env import coin_game
from utils.ttt_simple_env import ttt

# =============================================================================
# 1. THUẬT TOÁN MINIMAX ĐỆ QUY CHO TICTACTOE (Theo Chapter 5)
# =============================================================================
def maximized_payoff(env, reward, done):
    """
    Hàm đệ quy tính toán kết quả tốt nhất mà người chơi hiện tại có thể đạt được
    sau khi người chơi trước vừa thực hiện nước đi.
    """
    # 1. Điều kiện dừng: Ván cờ đã kết thúc
    if done:
        if reward != 0:
            return -1  # Đối thủ vừa đi xong và thắng -> Ta bị -1 điểm
        else:
            return 0   # Hòa cờ -> 0 điểm
            
    best_payoff = -2
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m)
        
        # Đệ quy: đối thủ sẽ phản ứng như thế nào ở lượt tiếp theo?
        opponent_payoff = maximized_payoff(env_copy, reward, done)
        
        # Lợi ích của ta là số đối của lợi ích đối thủ (Zero-Sum)
        my_payoff = -opponent_payoff
        
        if my_payoff > best_payoff:
            best_payoff = my_payoff
            
    return best_payoff

def MiniMax_TTT(env):
    """Hàm chọn nước đi tối ưu cho Tic Tac Toe bằng MiniMax"""
    wins = []
    ties = []
    losses = []
    
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m)
        
        # Nếu nước m giúp thắng ngay lập tức thì chọn luôn
        if done and abs(reward) == 1:
            return m
            
        opponent_payoff = maximized_payoff(env_copy, reward, done)
        my_payoff = -opponent_payoff
        
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)
            
    if len(wins) > 0:
        return random.choice(wins)
    elif len(ties) > 0:
        return random.choice(ties)
    return env.sample()

print("✅ Đã nạp môi trường coin_game và ttt thành công!")
print("✅ Thuật toán maximized_payoff và MiniMax_TTT đã sẵn sàng.")


## 2. Visualizer Cây Duyệt MiniMax (Dạng Đồ Họa Plot Trực Quan)

Dưới đây là sơ đồ cây đồ họa trực quan hóa nguyên lý **Quy nạp ngược (Backward Induction)** của MiniMax:
1. **Cây Coin Game (4 xu)**: Toàn bộ cây quyết định nhị phân từ gốc đến ngọn.
2. **Cây Tic Tac Toe (Thế tàn cuộc 3 ô trống: 4, 6, 8)**: Đánh giá cả 3 khả năng: Thắng (+1), Hòa (0), Thua (-1) và giải thích tại sao X chọn ô số 6.

In [ ]:
import matplotlib.pyplot as plt
from copy import deepcopy
from utils.ttt_simple_env import ttt

# =============================================================================
# 1. THIẾT KẾ THẾ CỜ TÀN HỢP LỆ TRONG TICTACTOE (ĐÃ ĐI 6 NƯỚC, CHƯA AI THẮNG)
# =============================================================================
# Chuỗi nước cờ chuẩn: X(5) -> O(1) -> X(9) -> O(2) -> X(3) -> O(7)
# Quân X chiếm: {3, 5, 9} | Quân O chiếm: {1, 2, 7}
# Các ô còn trống hợp lệ: 4, 6, 8. Lượt đi tiếp theo là: X
test_env = ttt()
test_env.reset()
for move in [5, 1, 9, 2, 3, 7]:
    test_env.step(move)

print("🎮 BÀN CỜ TICTACTOE THẾ MẪU (3x3):")
print(test_env.state.reshape(3, 3)[::-1])
print(f"- Các ô còn trống : {test_env.validinputs}")
print(f"- Lượt đánh kế    : Quân '{test_env.turn}' (MAX)\n")

# =============================================================================
# 2. HÀM HỖ TRỢ VẼ CÂY (TREE PLOTTER) BẰNG MATPLOTLIB
# =============================================================================
def draw_node(ax, x, y, text, fc, ec, lw=2, fontsize=8.5):
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.45', facecolor=fc, edgecolor=ec, linewidth=lw, alpha=0.95))

def draw_edge(ax, x1, y1, x2, y2, label="", color="#555555", lw=1.5, label_color="black", offset_x=0.0, offset_y=0.0):
    ax.annotate("", xy=(x2, y2 + 0.04), xytext=(x1, y1 - 0.04),
                arrowprops=dict(arrowstyle="->", lw=lw, color=color, mutation_scale=14, shrinkA=0))
    if label:
        mid_x = (x1 + x2) / 2 + offset_x
        mid_y = (y1 + y2) / 2 + offset_y
        ax.text(mid_x, mid_y, label, fontsize=8.5, fontweight='bold', color=label_color,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.85))

# =============================================================================
# 3. VẼ BỘ ĐÔI BIỂU ĐỒ CÂY MINIMAX: COIN GAME VS TICTACTOE
# =============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(21, 10.5))

# -----------------------------------------------------------------------------
# NHÁNH 1: CÂY QUYẾT ĐỊNH CHO COIN GAME (4 ĐỒNG XU)
# -----------------------------------------------------------------------------
ax1.set_title("🌳 CÂY MINIMAX: COIN GAME (4 ĐỒNG XU)\n(Mỗi lượt bốc 1 hoặc 2 xu, bốc xu cuối cùng là THẮNG)", 
              fontsize=12, fontweight="bold", pad=15)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')

# Gốc: 4 xu
draw_node(ax1, 0.50, 0.92, "GỐC: Còn 4 xu\n[MAX (Player 1)]\nĐịnh giá: +1", fc='#d4edda', ec='#28a745', lw=2.5)

# Tầng 1: Lượt MIN (Player 2)
draw_node(ax1, 0.26, 0.64, "Còn 3 xu\n[MIN (Player 2)]\nĐịnh giá: +1", fc='#fff3cd', ec='#ffc107', lw=2)
draw_node(ax1, 0.74, 0.64, "Còn 2 xu\n[MIN (Player 2)]\nĐịnh giá: -1", fc='#f8d7da', ec='#dc3545', lw=2)
draw_edge(ax1, 0.50, 0.92, 0.26, 0.64, label="Bốc 1 xu ★ (Tối ưu)", color="#28a745", lw=2.8, label_color="#1e7e34", offset_x=-0.04)
draw_edge(ax1, 0.50, 0.92, 0.74, 0.64, label="Bốc 2 xu", color="#888888", lw=1.5, offset_x=0.04)

# Tầng 2: Lượt MAX
draw_node(ax1, 0.13, 0.36, "Còn 2 xu\n[MAX (P1)]\nĐịnh giá: +1", fc='#d4edda', ec='#28a745')
draw_node(ax1, 0.39, 0.36, "Còn 1 xu\n[MAX (P1)]\nĐịnh giá: +1", fc='#d4edda', ec='#28a745')
draw_node(ax1, 0.61, 0.36, "Còn 1 xu\n[MAX (P1)]\nĐịnh giá: +1", fc='#d4edda', ec='#28a745')
draw_node(ax1, 0.87, 0.36, "Hết xu (0)\nMIN THẮNG (-1)\n(Lá cây)", fc='#f8d7da', ec='#dc3545', lw=2.5)

draw_edge(ax1, 0.26, 0.64, 0.13, 0.36, label="Bốc 1", color="#444444", offset_x=-0.02)
draw_edge(ax1, 0.26, 0.64, 0.39, 0.36, label="Bốc 2", color="#444444", offset_x=0.02)
draw_edge(ax1, 0.74, 0.64, 0.61, 0.36, label="Bốc 1", color="#444444", offset_x=-0.02)
draw_edge(ax1, 0.74, 0.64, 0.87, 0.36, label="Bốc 2 ★ (MIN thắng)", color="#dc3545", lw=2.2, label_color="#dc3545", offset_x=0.03)

# Tầng 3: Các lá MAX thắng
draw_node(ax1, 0.13, 0.08, "Hết xu (0)\nMAX THẮNG (+1)", fc='#cce5ff', ec='#007bff', lw=2.5)
draw_node(ax1, 0.39, 0.08, "Hết xu (0)\nMAX THẮNG (+1)", fc='#cce5ff', ec='#007bff', lw=2.5)
draw_node(ax1, 0.61, 0.08, "Hết xu (0)\nMAX THẮNG (+1)", fc='#cce5ff', ec='#007bff', lw=2.5)

draw_edge(ax1, 0.13, 0.36, 0.13, 0.08, label="Bốc 2 xu ★", color="#007bff", lw=2, label_color="#0056b3")
draw_edge(ax1, 0.39, 0.36, 0.39, 0.08, label="Bốc 1 xu ★", color="#007bff", lw=2, label_color="#0056b3")
draw_edge(ax1, 0.61, 0.36, 0.61, 0.08, label="Bốc 1 xu ★", color="#007bff", lw=2, label_color="#0056b3")

# -----------------------------------------------------------------------------
# NHÁNH 2: CÂY QUYẾT ĐỊNH CHO TICTACTOE (TÀN CUỘC CÒN Ô 4, 6, 8)
# -----------------------------------------------------------------------------
ax2.set_title("🌳 CÂY MINIMAX: TICTACTOE (TÀN CUỘC CÒN Ô 4, 6, 8)\n(Lượt X [MAX]: Đánh giá đầy đủ Thắng (+1), Hòa (0), Thua (-1))", 
              fontsize=12, fontweight="bold", pad=15)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')

# Gốc TicTacToe
draw_node(ax2, 0.50, 0.92, "GỐC: 3 ô trống (4, 6, 8)\nLượt: Quân 'X' [MAX]\nĐịnh giá MiniMax: +1", 
          fc='#d4edda', ec='#28a745', lw=2.5)

# Tầng 1: 3 Lựa chọn của X
draw_node(ax2, 0.18, 0.58, "X đi ô 6\nHoàn thành 3-6-9\nX THẮNG NGAY (+1)", fc='#cce5ff', ec='#007bff', lw=2.5)
draw_node(ax2, 0.50, 0.58, "X đi ô 4 (Chặn 1-4-7)\nLượt: O [MIN]\nĐịnh giá: 0 (Hòa)", fc='#e2e3e5', ec='#6c757d', lw=2)
draw_node(ax2, 0.82, 0.58, "X đi ô 8\nLượt: O [MIN]\nĐịnh giá: -1 (Thua)", fc='#f8d7da', ec='#dc3545', lw=2)

draw_edge(ax2, 0.50, 0.92, 0.18, 0.58, label="Đánh ô 6 ★ (Tối ưu)", color="#28a745", lw=2.8, label_color="#1e7e34", offset_x=-0.04)
draw_edge(ax2, 0.50, 0.92, 0.50, 0.58, label="Đánh ô 4", color="#888888", lw=1.5)
draw_edge(ax2, 0.50, 0.92, 0.82, 0.58, label="Đánh ô 8", color="#888888", lw=1.5, offset_x=0.04)

# Tầng 2: Phản ứng của O khi X đi ô 4 và ô 8
draw_node(ax2, 0.40, 0.18, "O đi ô 6 (chặn 3-6-9)\nSau đó X đi 8\nHÒA CỜ (0)", fc='#e2e3e5', ec='#6c757d', lw=2)
draw_node(ax2, 0.58, 0.18, "O đi ô 8\nSau đó X đi 6\nX THẮNG (+1)", fc='#cce5ff', ec='#007bff')
draw_edge(ax2, 0.50, 0.58, 0.40, 0.18, label="O đi 6 ★ (Hòa)", color="#444444", lw=2.0, offset_x=-0.03)
draw_edge(ax2, 0.50, 0.58, 0.58, 0.18, label="O đi 8", color="#aaaaaa", lw=1.2, offset_x=0.03)

draw_node(ax2, 0.74, 0.18, "O đi ô 4 (tạo 1-4-7)\nO THẮNG (-1)\n(X thua)", fc='#f8d7da', ec='#dc3545', lw=2.5)
draw_node(ax2, 0.90, 0.18, "O đi ô 6\nSau đó hòa cờ\nHÒA CỜ (0)", fc='#e2e3e5', ec='#6c757d')
draw_edge(ax2, 0.82, 0.58, 0.74, 0.18, label="O đi 4 ★ (Thắng)", color="#dc3545", lw=2.2, label_color="#dc3545", offset_x=-0.03)
draw_edge(ax2, 0.82, 0.58, 0.90, 0.18, label="O đi 6", color="#aaaaaa", lw=1.2, offset_x=0.03)

plt.tight_layout()
plt.show()

print("💡 BÀI HỌC TỪ CÂY ĐỒ HỌA:")
print("1. Ở Coin Game: MAX chọn bốc 1 xu vì nhánh 1 xu dẫn tới +1 ở cả 2 phản ứng của MIN, trong khi nhánh 2 xu bị MIN trừng phạt bằng -1.")
print("2. Ở TicTacToe: MAX (X) chọn đánh ô 6 vì mang lại chiến thắng ngay lập tức (+1), tránh rủi ro bị hòa (ô 4) hoặc bị thua (ô 8).")


## 3. Tỉa Cây Theo Các Phương Pháp (Depth Pruning & Alpha-Beta Pruning)

Khi bàn cờ còn trống, cây tìm kiếm Tic Tac Toe có tới $9! = 362.880$ nhánh, chạy vét cạn mất 30-40 giây. Ta có 2 giải pháp tỉa cây kinh điển:
1. **Cắt tỉa độ sâu (Depth Pruning - Chapter 5)**: Giới hạn độ sâu tối đa $d$. Nếu chưa hết ván thì tạm coi là Hòa (0 điểm).
2. **Cắt tỉa Alpha-Beta (Alpha-Beta Pruning - Chapter 6)**: Loại bỏ các nhánh chắc chắn không ảnh hưởng đến kết quả tối ưu, giảm hơn 90% số phép tính mà **không làm mất tính chính xác**.

In [ ]:
# Biến đếm số lần duyệt nút để so sánh hiệu năng giữa các phương pháp
node_count_full = 0
node_count_depth = 0
node_count_ab = 0

# =============================================================================
# 1. MINIMAX THUẦN TÚY (VÉT CẠN TOÀN BỘ)
# =============================================================================
def minimax_full(env, is_max):
    global node_count_full
    node_count_full += 1
    
    if env.done:
        return env.reward
        
    if is_max:
        best = -float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_full(env_copy, False)
            best = max(best, val)
        return best
    else:
        best = float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_full(env_copy, True)
            best = min(best, val)
        return best

# =============================================================================
# 2. CẮT TỈA ĐỘ SÂU (DEPTH PRUNING - CHAPTER 5)
# =============================================================================
def minimax_depth_pruning(env, depth, max_depth, is_max):
    global node_count_depth
    node_count_depth += 1
    
    if env.done:
        return env.reward
    # [ĐIỂM DỪNG MỚI]: Dừng lại khi chạm ngưỡng max_depth
    if depth >= max_depth:
        return 0  # Heuristic tạm thời: Coi như hòa nếu chưa xong
        
    if is_max:
        best = -float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_depth_pruning(env_copy, depth + 1, max_depth, False)
            best = max(best, val)
        return best
    else:
        best = float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_depth_pruning(env_copy, depth + 1, max_depth, True)
            best = min(best, val)
        return best

# =============================================================================
# 3. CẮT TỈA ALPHA-BETA (ALPHA-BETA PRUNING - CHAPTER 6)
# =============================================================================
def minimax_alpha_beta(env, alpha, beta, is_max):
    global node_count_ab
    node_count_ab += 1
    
    if env.done:
        return env.reward
        
    if is_max:
        best = -float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_alpha_beta(env_copy, alpha, beta, False)
            best = max(best, val)
            alpha = max(alpha, best)
            # [CẮT TỈA]: Đối thủ ở tầng trên chắc chắn không bao giờ cho phép ta chọn nhánh này
            if beta <= alpha:
                break
        return best
    else:
        best = float('inf')
        for m in env.validinputs:
            env_copy = deepcopy(env)
            env_copy.step(m)
            val = minimax_alpha_beta(env_copy, alpha, beta, True)
            best = min(best, val)
            beta = min(beta, best)
            # [CẮT TỈA]: Ta ở tầng trên đã có lựa chọn tốt hơn, bỏ qua các nhánh còn lại
            if beta <= alpha:
                break
        return best

# =============================================================================
# THỬ NGHIỆM ĐO LƯỜNG HIỆU NĂNG TRÊN THẾ CỜ CÒN 5 Ô TRỐNG
# =============================================================================
benchmark_env = ttt()
benchmark_env.reset()
# Đánh 4 nước: X đi 1, O đi 5, X đi 9, O đi 3 (Còn 5 ô trống: 2, 4, 6, 7, 8)
for m in [1, 5, 9, 3]:
    benchmark_env.step(m)

print("=" * 75)
print("⚡ SO SÁNH SỐ NÚT DUYỆT (TỐC ĐỘ) GIỮA 3 PHƯƠNG PHÁP TRÊN THẾ CỜ 5 Ô TRỐNG:")
print("=" * 75)

# 1. Chạy Full MiniMax
node_count_full = 0
t0 = time.time()
score_full = minimax_full(deepcopy(benchmark_env), is_max=True)
time_full = time.time() - t0

# 2. Chạy Depth Pruning (max_depth = 2)
node_count_depth = 0
t0 = time.time()
score_depth = minimax_depth_pruning(deepcopy(benchmark_env), depth=0, max_depth=2, is_max=True)
time_depth = time.time() - t0

# 3. Chạy Alpha-Beta Pruning
node_count_ab = 0
t0 = time.time()
score_ab = minimax_alpha_beta(deepcopy(benchmark_env), alpha=-float('inf'), beta=float('inf'), is_max=True)
time_ab = time.time() - t0

import pandas as pd
df_comparison = pd.DataFrame({
    "Phương pháp": ["MiniMax Thuần túy (Full)", "Cắt tỉa độ sâu (Depth Pruning d=2)", "Cắt tỉa Alpha-Beta (Alpha-Beta)"],
    "Định giá": [score_full, score_depth, score_ab],
    "Số nút đã duyệt": [node_count_full, node_count_depth, node_count_ab],
    "Thời gian (giây)": [round(time_full, 4), round(time_depth, 4), round(time_ab, 4)],
    "Tốc độ giảm tải (%)": [
        "Gốc (100%)",
        f"Giảm {((node_count_full - node_count_depth) / node_count_full * 100):.1f}% nút",
        f"Giảm {((node_count_full - node_count_ab) / node_count_full * 100):.1f}% nút (Tối ưu 100%)"
    ]
})

display(df_comparison)
print("\n💡 Nhận xét: Alpha-Beta Pruning giữ nguyên 100% kết quả tối ưu nhưng giảm mạnh số nút cần duyệt!")


## 📌 Tóm Tắt & Bài Học Cốt Lõi

1. **Coin Game** là mô hình trực quan hoàn hảo nhất để hiểu MiniMax: Bằng cách đi lùi từ trạng thái 0 xu (Backward Induction), ta xác định được người đi trước sẽ thắng nếu bốc số xu thích hợp.
2. **Đệ quy trong Tic Tac Toe** thay thế toàn bộ các vòng lặp lồng nhau của Chương 2 (`think1`, `think2`, `think3`), tự động nhìn xa vô hạn bước tới cuối ván cờ.
3. **Cắt tỉa độ sâu (Depth Pruning)** giúp giải quyết bài toán bùng nổ tổ hợp khi bàn cờ quá lớn (như Connect Four hay Cờ vua), trong khi **Cắt tỉa Alpha-Beta** loại bỏ các nhánh thừa mà vẫn bảo toàn độ chính xác tuyệt đối.